In [ ]:
# Basic setup
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import ipywidgets as widgets
from IPython.display import display, clear_output

pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

print("Libraries loaded.")

In [ ]:
# Load your file (upload job_market.csv first)
df = pd.read_csv("job_market.csv")

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

In [ ]:
# Rename columns
df = df.rename(columns={
    'Job Title': 'title',
    'Location': 'location',
    'Salary': 'salary_text',
    'City': 'city',
    'State': 'state'
})

# Tamil Nadu / Chennai filter
df['is_tn'] = df['location'].str.contains(
    r'chen|chennai|tamil|coimbatore|madurai|salem|tn', case=False, na=False
) | df['city'].str.contains('chen|chennai', case=False, na=False)

print("Rows in Tamil Nadu:", df['is_tn'].sum())

# Keep only IT / software / fresher related jobs
it_pattern = r'(?i)(software|developer|engineer|python|java|sql|data|analyst|tester|aws|trainee|intern|fresher|it)'
df_it = df[df['title'].str.contains(it_pattern, na=False) & df['is_tn']].reset_index(drop=True)

print("\nFinal IT jobs in TN:", len(df_it))
display(df_it[['title', 'location', 'salary_text']].head(6))

In [ ]:
# Simple skill extraction from title
def get_skills(title):
    text = str(title).lower()
    skills = []
    if 'python' in text: skills.append('Python')
    if 'java' in text:   skills.append('Java')
    if 'sql' in text:    skills.append('SQL')
    if 'aws' in text:    skills.append('AWS')
    if 'data' in text:   skills.append('Data')
    if 'test' in text or 'qa' in text: skills.append('Testing')
    if 'full' in text and 'stack' in text: skills.append('Full Stack')
    if not skills: skills.append('General IT')
    return ', '.join(skills)

df_it['skills'] = df_it['title'].apply(get_skills)

print("Top skills:")
print(df_it['skills'].value_counts().head(8))

# Salary → LPA (very simple version)
def salary_to_lpa(s):
    if pd.isna(s) or 'not' in str(s).lower():
        return np.nan
    nums = re.findall(r'\d+\.?\d*', str(s))
    if not nums: return np.nan
    val = float(nums[0])
    if val < 10000: return val * 12 / 100000  # monthly
    elif val <= 100000: return val * 12 / 100000
    else: return val / 100000

df_it['salary_lpa'] = df_it['salary_text'].apply(salary_to_lpa)
df_it['salary_lpa'] = df_it['salary_lpa'].fillna(df_it['salary_lpa'].median(skipna=True))

print("\nSalary (LPA) summary:")
print(df_it['salary_lpa'].describe().round(2))

In [ ]:
# Graph 1: Salary distribution
plt.figure(figsize=(9,5))
sns.histplot(df_it['salary_lpa'], bins=15, kde=True, color='teal')
plt.title("Salary Distribution – IT Fresher Jobs in TN")
plt.xlabel("Estimated LPA")
plt.show()

# Graph 2: Top locations
top_loc = df_it['location'].value_counts().head(8)
fig = px.bar(x=top_loc.values, y=top_loc.index, orientation='h',
             title="Top Locations in Tamil Nadu")
fig.show()

# Graph 3: Top skill combinations
top_sk = df_it['skills'].value_counts().head(8)
fig2 = px.bar(x=top_sk.values, y=top_sk.index, orientation='h',
              title="Most Common Skill Mentions")
fig2.show()

In [ ]:
# ────────────────────────────────────────────────
#   Prepare TF-IDF for skill similarity matching
# ────────────────────────────────────────────────

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Use clean version without the warning
vectorizer = TfidfVectorizer(
    tokenizer=lambda x: [s.strip() for s in x.split(',') if s.strip()],
    token_pattern=None,           # we handle splitting ourselves
    lowercase=True,
    stop_words=None,              # ← important: disables stop_words → removes warning
    max_features=100              # optional: limits vocabulary size
)

# Fit on the skills column
tfidf_matrix = vectorizer.fit_transform(df_it['skills'].fillna('General IT'))

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(vectorizer.get_feature_names_out()))
print("\nExample features:", vectorizer.get_feature_names_out()[:15])

In [ ]:
# ==================================================
#   SUPER CLEAN OUTPUT - EXACTLY AS YOU WANT
# ==================================================

import ipywidgets as widgets
from IPython.display import display, clear_output

# Simple mapping: current skill → most realistic next skill (only 1)
realistic_next = {
    'python': 'AWS Basics',
    'sql': 'Power BI',
    'java': 'Spring Boot',
    'c': 'Python',
    'html': 'React Basics',
    'general it': 'Python',
}

# ─── Widgets ────────────────────────────────────────────────
qual = widgets.Dropdown(
    options=['B.Tech / BE', 'B.Sc CS/IT', 'BCA', 'M.Sc / MCA', 'Diploma', 'Other'],
    description='Qualification:',
)

gap = widgets.IntSlider(
    min=0, max=10, step=1, value=1,
    description='Career Gap (years):',
)

current = widgets.Textarea(
    value='Python, SQL',
    placeholder='comma separated, e.g. Python, Java, SQL',
    description='Current Skills:',
    rows=2
)

btn = widgets.Button(
    description="Get Recommendation",
    button_style='success'
)

output = widgets.Output()

# ─── Clean logic ────────────────────────────────────────────
def on_click(b):
    with output:
        clear_output()

        q = qual.value
        g = gap.value
        curr_raw = current.value.strip()
        curr_list = [s.strip().lower() for s in curr_raw.split(',') if s.strip()]

        if not curr_list:
            print("Please enter at least 1 current skill.")
            return

        # Pick one realistic next skill
        next_skill = 'Python'  # default
        for skill in curr_list:
            if skill in realistic_next:
                next_skill = realistic_next[skill]
                break

        # Salary logic
        base = 4.8
        if 'M.Sc' in q or 'MCA' in q or 'B.Tech' in q:
            base += 0.8
        gap_reduce = g * 0.35
        est_min = round(max(3.2, base - gap_reduce - 0.6), 1)
        est_max = round(base - gap_reduce + 1.0, 1)

        # Job based on next skill
        job = 'Software / IT Trainee'
        if 'AWS' in next_skill: job = 'Cloud Support Trainee'
        elif 'Power BI' in next_skill or 'Data' in next_skill: job = 'Junior Data Analyst'
        elif 'Spring' in next_skill: job = 'Java Backend Developer'
        elif 'Python' in next_skill: job = 'Python Developer'

        # Final clean output - no extra tags, no numbers
        print(f"Qualification: {q}")
        print(f"Career Gap: {g} year{'s' if g != 1 else ''}")
        print(f"Current Skills: {', '.join([s.title() for s in curr_list])}")
        print()
        print(f"Recommended skill to learn: {next_skill}")
        print()
        print(f"Possible Job Roles: {job}")
        print()
        print(f"Expected Salary Range in Chennai: ₹ {est_min} – {est_max} LPA")
        print("(Lower if gap is longer, higher with good interview & company)")

        if g >= 4:
            print("\nNote: With longer gap → finish this skill + 1 small project to explain gap better.")

btn.on_click(on_click)

display(qual, gap, current, btn, output)